In [182]:
import os

import importlib
import json
import pandas as pd
import random
import time

from datetime import date, datetime
from dotenv import load_dotenv
from openai import OpenAI

import parsing
import evaluation
import exporting

from parsing import parse_output
from evaluation import evaluation_v1, evaluation_v2, evaluation_v3
from exporting import export_run

In [183]:
random.seed(42)

In [184]:
importlib.reload(parsing)
from parsing import parse_output

importlib.reload(evaluation)
from evaluation import evaluation_v1, evaluation_v2, evaluation_v3, evaluation_v4

importlib.reload(exporting)
from exporting import export_run

In [185]:
EVALUATORS = {
    "v1": evaluation_v1,
    "v2": evaluation_v2,
    "v3": evaluation_v3,
    "v4": evaluation_v4
}

# BioRED Train & GT Loading & Parsing

### Load Key

In [5]:
load_dotenv()
print(os.getenv("OPEN_AI_TEST_KEY")[:15])

sk-proj-4WDSBIA


In [6]:
client = OpenAI(
    api_key=os.getenv("OPEN_AI_TEST_KEY")
)

### Load & Downsample BioRED Train Abstracts

In [7]:
biored_train = pd.read_csv("../data/processed/biored/br_train.csv")

biored_train_sample = biored_train.sample(
    n=35,
    random_state=42
).copy()

### BioRED GT Parsing

In [ ]:
sampled_pmids = set(biored_train_sample["pmid"])

biored_train_gts = pd.read_csv("../data/processed/biored/br_train_entity_relations.csv")

biored_train_gts_filtered = biored_train_gts[biored_train_gts["pmid"].isin(sampled_pmids)]

### Export Filtered GTS

In [206]:
os.makedirs("../data/filtered", exist_ok=True)

biored_train_gts_filtered.to_csv("../data/filtered/biored_train_gts_filtered.csv")

### Parse Into Two Sets of Tuples
* `predictions_entities`
* `predictions_relationships`

**Includes PMID for duplicate handling as sets automatically remove duplicates**
* PMID avoids duplicates across papers

In [9]:
ground_truth_entities = (
    set(zip(biored_train_gts_filtered["pmid"], biored_train_gts_filtered["entity_1"].str.strip().str.lower(), biored_train_gts_filtered["entity_1_type"]))
    | set(zip(biored_train_gts_filtered["pmid"], biored_train_gts_filtered["entity_2"].str.strip().str.lower(), biored_train_gts_filtered["entity_2_type"]))
)

ground_truth_relationships = set(zip(
    biored_train_gts_filtered["pmid"],
    biored_train_gts_filtered["entity_1"].str.strip().str.lower(),
    biored_train_gts_filtered["relation"],
    biored_train_gts_filtered["entity_2"].str.strip().str.lower()
))

In [10]:
ground_truths = {
    "entities": ground_truth_entities,
    "relationships": ground_truth_relationships
}

### Few-Shot Construction

In [207]:
# Exclude the 35 pmids you're evaluating on
non_eval_pmids = set(biored_train["pmid"]) - sampled_pmids
few_shot_pmids = random.sample(list(non_eval_pmids), 3)

def build_few_shot_example(pmid, example_num):
    abstract = biored_train[biored_train["pmid"] == pmid]["abstract"].iloc[0]
    gt_rows = biored_train_gts[biored_train_gts["pmid"] == pmid]

    entities = list({
        (row["entity_1"], row["entity_1_type"]) for _, row in gt_rows.iterrows()
    } | {
        (row["entity_2"], row["entity_2_type"]) for _, row in gt_rows.iterrows()
    })

    relationships = [
        {"source": row["entity_1"], "relation": row["relation"], "target": row["entity_2"]}
        for _, row in gt_rows.iterrows()
    ]

    output_json = {
        "entities": [{"text": e[0], "type": e[1]} for e in entities],
        "relationships": relationships
    }

    return f"## EXAMPLE {example_num+1}:\n\n### Abstract:\n\n{abstract}\n\n### Correct BioRED annotation:\n\n{json.dumps(output_json, indent=2)}"

few_shot_block = "\n\n".join(build_few_shot_example(pmid, i) for i, pmid in enumerate(few_shot_pmids))

# print(few_shot_block[:1750])  # sanity check

### Import BioRED Extraction Guidelines from `guidelines.txt`

In [216]:
with open("../data/processed/biored/guidelines.txt", "r", encoding="utf-8") as f:
    biored_ext_guidelines = f.read()

biored_ext_guidelines[:100]

'## Guideline of the entities\n\n### General rules\n- Annotate all the spans of all the six concept type'

# LLM API Call

In [218]:
with open("../data/prompt_refinement/prompt_versions.json", "r") as f:
    PROMPTS = json.load(f)

In [219]:
def load_run_config(run_name, path="run_configs.json"):
    with open(path) as f:
        configs = json.load(f)
    if run_name not in configs:
        raise KeyError(f"No config found for '{run_name}'")
    return configs[run_name]

In [220]:
RUN = "002"

cfg = load_run_config(f"run{RUN}")
PROMPT_VERSION = cfg["prompt_version"]
EVAL_VERSION = cfg["eval_version"]
RUN_NOTES = cfg["notes"]

print(f"Last ran at {datetime.now().strftime('%Y-%m-%d %H:%M')} for run{RUN}")
print(f" - Prompt verion: {PROMPT_VERSION}")
print(f" - Evalaution verion: {EVAL_VERSION}")
print(f" - Run notes: '{RUN_NOTES}'")

Last ran at 2026-07-29 14:03 for run002
 - Prompt verion: v2
 - Evalaution verion: v1
 - Run notes: 'Implemented BioRED entity and relation schema and rules.'


In [221]:
start_time = time.perf_counter()

outputs = []

for index, row in biored_train_sample.iterrows():
    abstract = row["abstract"]

    prompt = (
        PROMPTS[PROMPT_VERSION]["template"]
        .replace("{abstract}", abstract)
        .replace("{few_shot_block}", few_shot_block)
        .replace("{biored_ext_guidelines}", biored_ext_guidelines)
    )

    response = client.responses.create(
        model="gpt-5.6-luna",
        input=prompt
    )

    outputs.append({
        "pmid": row["pmid"],
        "output": response.output_text
    })

elapsed_seconds = time.perf_counter() - start_time
print(f"API calls took {elapsed_seconds:.1f}s ({elapsed_seconds/60:.1f} min) for {len(outputs)} abstracts")

print(f"Successfully ran API call at {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f" - Prompt verion: {PROMPT_VERSION}")
print(f" - Evalaution verion: {EVAL_VERSION}")
print(f" - Run notes: {RUN_NOTES}")
print(f" - Elapsed time: {elapsed_seconds:.1f}s ({elapsed_seconds/60:.1f}")

API calls took 615.9s (10.3 min) for 35 abstracts
Successfully ran API call at 2026-07-29 14:13
 - Prompt verion: v2
 - Evalaution verion: v1
 - Run notes: Implemented BioRED entity and relation schema and rules.
 - Elapsed time: 615.9s (10.3


In [233]:
output_info = {
    "outputs": outputs,
    "run_notes": RUN_NOTES,
    "prompt_version": PROMPT_VERSION,
    "eval_version": EVAL_VERSION
}

### Parse `outputs` list to JSON

In [234]:
run_info = parse_output(output_info)

Parsed 35 extractions, 0 failed to parse as JSON


# Evaluation Metrics

In [235]:
evaluator = EVALUATORS[EVAL_VERSION]
eval_results = evaluator(run_info, ground_truths)

# Export Run

### Create Run Storage Dir.

In [236]:
os.makedirs("../data/exploration", exist_ok=True)

In [237]:
raw_prompt = PROMPTS[PROMPT_VERSION]["template"]

In [238]:
export_run(run_info, eval_results, eval_version=EVAL_VERSION, prompt_version=PROMPT_VERSION, prompt=raw_prompt, time_taken=elapsed_seconds);

Saved run_002 to ../prompt_runs


# Error Sampling

In [239]:
def sample_errors_strict(predictions, ground_truth, n=20, label="items"):
    false_positives = list(predictions - ground_truth)
    false_negatives = list(ground_truth - predictions)

    fp_sample = random.sample(false_positives, min(n, len(false_positives)))
    fn_sample = random.sample(false_negatives, min(n, len(false_negatives)))

    print(f"--- {label}: False Positives (predicted, not in ground truth) ---")
    print(f"Sampled {len(fp_sample)} of {len(false_positives)} total FPs\n")
    for item in fp_sample:
        print(" ", item)

    print(f"\n--- {label}: False Negatives (in ground truth, not predicted) ---")
    print(f"Sampled {len(fn_sample)} of {len(false_negatives)} total FNs\n")
    for item in fn_sample:
        print(" ", item)

    return fp_sample, fn_sample

In [240]:
from evaluation import COSINE_THRESHOLD

def sample_errors_cosine(predictions, ground_truth, match_fn, embeddings, threshold=COSINE_THRESHOLD, n=20, label="items"):
    matched_predictions = set()
    matched_gt = set()

    for p in predictions:
        best_score = -1
        best_g = None

        for g in ground_truth:
            if g in matched_gt:
                continue
            is_match, score = match_fn(p, g, embeddings, threshold)
            if is_match and score > best_score:
                best_score = score
                best_g = g

        if best_g is not None:
            matched_gt.add(best_g)
            matched_predictions.add(p)

    false_positives = list(predictions - matched_predictions)
    false_negatives = list(ground_truth - matched_gt)

    fp_sample = random.sample(false_positives, min(n, len(false_positives)))
    fn_sample = random.sample(false_negatives, min(n, len(false_negatives)))

    print(f"--- {label}: False Positives (predicted, no cosine match in ground truth) ---")
    print(f"Sampled {len(fp_sample)} of {len(false_positives)} total FPs\n")
    for item in fp_sample:
        print(" ", item)

    print(f"\n--- {label}: False Negatives (in ground truth, no cosine match in predictions) ---")
    print(f"Sampled {len(fn_sample)} of {len(false_negatives)} total FNs\n")
    for item in fn_sample:
        print(" ", item)

    return fp_sample, fn_sample

In [241]:
from evaluation import relationship_match_cosine, entity_match_cosine, build_embedding_lookup

relationship_embeddings = build_embedding_lookup(
    run_info["predictions_relationships"], ground_truth_relationships, text_indices=[1, 3]
)

In [242]:
relation_fp_sample, relation_fn_sample = sample_errors_cosine(
    run_info["predictions_relationships"],
    ground_truth_relationships,
    relationship_match_cosine,
    relationship_embeddings,
    n=100,
    label="Relationships"
)

--- Relationships: False Positives (predicted, no cosine match in ground truth) ---
Sampled 100 of 133 total FPs

  (19891556, 'clopidogrel', 'Negative_Correlation', 'coronary artery disease')
  (24477591, 'reactive oxygen species', 'Positive_Correlation', 'atherosclerosis')
  (18808529, 'dystrophin', 'Bind', 'laminin')
  (15099351, 'pcsk9', 'Association', 'd374y')
  (19521089, '5-httlpr', 'Association', 'slc6a4')
  (29222418, 'id3', 'Negative_Correlation', 'sox4')
  (25305591, 'il-10', 'Negative_Correlation', 'experimental autoimmune encephalomyelitis')
  (28428256, 'lps', 'Positive_Correlation', 'vcam1')
  (24341598, 'isotonic sodium chloride', 'Negative_Correlation', 'contrast-induced nephropathy')
  (28428256, 'pga2', 'Negative_Correlation', 'thrombin')
  (28411266, 'abcc8', 'Association', 'type 2 diabetes')
  (16288197, 'q48h', 'Association', 'glaucoma')
  (18808529, 'isoproterenol', 'Negative_Correlation', 'laminin alpha-2')
  (19521089, 'slc6a4', 'Positive_Correlation', "parkins

In [243]:
entity_embeddings = build_embedding_lookup(
    run_info["predictions_entities"], ground_truth_entities, text_indices=[1]
)

entity_fp_sample, entity_fn_sample = sample_errors_cosine(
    run_info["predictions_entities"],
    ground_truth_entities,
    entity_match_cosine,
    entity_embeddings,
    n=100,
    label="Entities"
)

--- Entities: False Positives (predicted, no cosine match in ground truth) ---
Sampled 100 of 459 total FPs

  (24477591, 'coronary artery disease', 'Disease')
  (17192049, 'ile462val', 'Variant')
  (18768591, 'serum- and glucocorticoid-inducible kinase', 'Gene')
  (16200390, '5-hydroxytryptamine', 'Chemical')
  (19521089, 'serotonin transporter gene', 'Gene')
  (18768591, 'hypoalbuminemia', 'Disease')
  (24309294, 'rats', 'Species')
  (24914936, 'thyroid hormone', 'Chemical')
  (20431083, 'tia', 'Disease')
  (20431083, 'ischemic stroke', 'Disease')
  (25305591, 'pituitary adenylyl cyclase-activating polypeptide', 'Chemical')
  (19319147, 'intracerebral hemorrhage', 'Disease')
  (24914936, 'developmental delay', 'Disease')
  (18768591, 'renal fibrosis', 'Disease')
  (19521089, 'insertion/deletion (long allele-l/short allele-s) of 43 bp', 'Variant')
  (24914936, 'skeletal dysplasia', 'Disease')
  (24477591, 'premature cad', 'Disease')
  (24914936, 'thra', 'Gene')
  (24477591, 'ros', 'Ch